In [6]:
import numpy as np

In [7]:
# Problem 0:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS = 3
NUM_ITEMS = 10
NUM_DIMENSIONS = 2
VALUES = rng.integers(0, 100, size=NUM_ITEMS)
WEIGHTS = rng.integers(0, 100, size=(NUM_ITEMS, NUM_DIMENSIONS))
CONSTRAINTS = rng.integers(
    0, 100 * NUM_ITEMS // NUM_KNAPSACKS, size=(NUM_KNAPSACKS, NUM_DIMENSIONS)
)

In [8]:
# Problem 1:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS1 = 3
NUM_ITEMS1 = 20
NUM_DIMENSIONS1 = 2
VALUES1 = rng.integers(0, 100, size=NUM_ITEMS1)
WEIGHTS1 = rng.integers(0, 100, size=(NUM_ITEMS1, NUM_DIMENSIONS1))
CONSTRAINTS1 = rng.integers(
    0, 100 * NUM_ITEMS1 // NUM_KNAPSACKS1, size=(NUM_KNAPSACKS1, NUM_DIMENSIONS1)
)

In [9]:
# Problem 2:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS2 = 10
NUM_ITEMS2 = 100
NUM_DIMENSIONS2 = 10
VALUES2 = rng.integers(0, 1000, size=NUM_ITEMS2)
WEIGHTS2 = rng.integers(0, 1000, size=(NUM_ITEMS2, NUM_DIMENSIONS2))
CONSTRAINTS2 = rng.integers(
    1000 * 2, 1000 * NUM_ITEMS2 // NUM_KNAPSACKS2, size=(NUM_KNAPSACKS2, NUM_DIMENSIONS2)
)

In [10]:
# Problem 3:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS3 = 100
NUM_ITEMS3 = 5000
NUM_DIMENSIONS3 = 100
VALUES3 = rng.integers(0, 1000, size=NUM_ITEMS3)
WEIGHTS3 = rng.integers(0, 1000, size=(NUM_ITEMS3, NUM_DIMENSIONS3))
CONSTRAINTS3 = rng.integers(
    1000 * 10, 1000 * 2 * NUM_ITEMS3 // NUM_KNAPSACKS3, size=(NUM_KNAPSACKS3, NUM_DIMENSIONS3)
)

In [11]:
def is_valid(solution, num_knapsacks, weights, constraints):
    if not np.all(solution.sum(axis=0) <= 1):
        return False

    for k in range(num_knapsacks):
        items_in_knapsack_k = solution[k]
        weight_of_knapsack_k = weights[items_in_knapsack_k].sum(axis=0)
        if not np.all(weight_of_knapsack_k <= constraints):
            return False

    return True

In [12]:
def evaluate(solution, values, num_knapsacks, weights, constraints):
    if not is_valid(solution, num_knapsacks, weights, constraints):
        return -1.0
    items_placed = np.any(solution, axis=0)
    total_value = values[items_placed].sum()
    return float(total_value)

In [13]:
def move(solution, num_items, num_knapsacks):
    neighbor = solution.copy()
    item_to_move = rng.integers(0, num_items)
    new_knapsack_idx = rng.integers(-1, num_knapsacks)
    neighbor[:, item_to_move] = False
    if new_knapsack_idx != -1:
        neighbor[new_knapsack_idx, item_to_move] = True

    return neighbor

In [14]:
def hill_climber_fast(initial_solution: np.ndarray, max_steps_no_improvement: int, values, num_knapsacks, weights,
                      constraints, num_items):
    current_solution, current_score = initial_solution, evaluate(initial_solution, values, num_knapsacks, weights,
                                                                 constraints)
    steps_without_improvement = 0
    while steps_without_improvement < max_steps_no_improvement:
        neighbor = move(current_solution, num_items, num_knapsacks)
        neighbor_score = evaluate(neighbor, values, num_knapsacks, weights, constraints)
        if neighbor_score > current_score:
            current_solution, current_score = neighbor, neighbor_score
            steps_without_improvement = 0
        else:
            steps_without_improvement += 1
    return current_solution, current_score

In [15]:
def create_random_valid_solution(num_knapsacks, num_items, weights, constraints):
    solution = np.zeros((num_knapsacks, num_items), dtype=bool)
    shuffled_items = list(range(num_items))
    rng.shuffle(shuffled_items)

    for item_idx in shuffled_items:
        knapsack_idx = rng.integers(0, num_knapsacks)
        solution[knapsack_idx, item_idx] = True
        if not is_valid(solution, num_knapsacks, weights, constraints):
            solution[knapsack_idx, item_idx] = False

    return solution

In [16]:
def crossover(parent1: np.ndarray, parent2: np.ndarray, num_items) -> np.ndarray:
    child = np.zeros_like(parent1)
    for i in range(num_items):
        if rng.random() < 0.5:
            child[:, i] = parent1[:, i]
        else:
            child[:, i] = parent2[:, i]
    return child

In [17]:
def simulated_annealing_fast(initial_solution: np.ndarray, max_steps: int, initial_temp: float, cooling_rate: float, values, num_knapsacks, weights, constraints, num_items):
    """
    fast version of SA to be used as local search engine.
    """
    current_solution = initial_solution
    current_score = evaluate(current_solution, values, num_knapsacks, weights, constraints)
    best_solution, best_score = current_solution, current_score
    temperature = initial_temp

    for _ in range(max_steps):
        neighbor = move(current_solution, num_items, num_knapsacks)
        neighbor_score = evaluate(neighbor, values, num_knapsacks, weights, constraints)

        #accept always if better, else with a probability
        if neighbor_score > current_score or rng.random() < np.exp((neighbor_score - current_score) / temperature):
            current_solution, current_score = neighbor, neighbor_score

        # update best solution found
        if current_score > best_score:
            best_solution, best_score = current_solution, current_score

        temperature *= cooling_rate
        if temperature < 1e-3:  # avoid too low temperatures
            break

    return best_solution, best_score

In [18]:
# combines the principles of evolution (recombination and selection) with those of individual learning (local search). 
# It balances global exploration (genetic diversity) with local  intensification (solution refinement)
def my_algorithm(generations: int, mu: int, lambda_: int, num_knapsacks, num_items, weights, constraints, values):
    # INITIALIZATION:
    #    - A population of mu feasible random solutions is generated, where each solution encodes an assignment of items to knapsacks
    population = [create_random_valid_solution(num_knapsacks, num_items, weights, constraints) for _ in range(mu)]
    # SOL FOR PROB3: population = [np.zeros((NUM_KNAPSACKS, NUM_ITEMS), dtype=bool) for _ in range(mu)]
    best_solution_so_far, best_score_so_far = None, -1
    # EVOLUTIONARY LOOP 
    for gen in range(generations):
        offspring = []
        for _ in range(lambda_):
            # offspring are created by randomly selecting parent pairs.
            parent1 = population[rng.integers(0, mu)]
            parent2 = population[rng.integers(0, mu)]
            # The crossover operator mixes the parents’ item assignments to produce new offspring inheriting traits from both parents.
            child = crossover(parent1, parent2, num_items)
            # A mutation operator makes small random changes to the offspring solutions.
            child = move(child, num_items, num_knapsacks)
            #optimize child with SA
            # LOCAL SEARCH (memetic phase):
            #- Each offspring is refined using simulated annealing that performs small modifications and occasionally accepts worse solutions to escape local optima. (This allows each individual to "learn" or improve on its own.)
            improved_child, _ = simulated_annealing_fast(child, max_steps=75, initial_temp=10.0, cooling_rate=0.99, values=values, num_knapsacks=num_knapsacks, weights=weights, constraints=constraints, num_items=num_items)
            offspring.append(improved_child)
            # SELECTION:
            #- Parents and offspring are merged into a (mu + lambda) population.
            #- All individuals are evaluated via the objective function.
            # - The mu best solutions are kept for the next generation, ensuring selective pressure toward higher-quality individuals.
        combined_population = population + offspring
        scores = [evaluate(ind, values, num_knapsacks, weights, constraints) for ind in combined_population]
        sorted_indices = np.argsort(scores)[::-1]
        population = [combined_population[i] for i in sorted_indices[:mu]]

        current_best_score = scores[sorted_indices[0]]
        if current_best_score > best_score_so_far:
            best_score_so_far = current_best_score
            best_solution_so_far = population[0]
            print(f"  GENERATION {gen + 1}: new record! value = {best_score_so_far}")

    return best_solution_so_far, best_score_so_far

In [19]:
#PROBLEM1
GENERATIONS = 50
POPULATION_SIZE = 20  # mu (dim of population)
OFFSPRING_SIZE = 200  # lambda (num of children per generation)

best_solution, best_score = my_algorithm(
    generations=GENERATIONS,
    mu=POPULATION_SIZE,
    lambda_=OFFSPRING_SIZE, 
    num_knapsacks=NUM_KNAPSACKS1,
    num_items=NUM_ITEMS1, 
    weights=WEIGHTS1, 
    constraints=CONSTRAINTS1, 
    values=VALUES1
)

if best_score <= 0:
    print("no solution found")
else:
    print("Value:", best_score)

  GENERATION 1: new record! value = 776.0
  GENERATION 2: new record! value = 804.0
  GENERATION 3: new record! value = 855.0
  GENERATION 7: new record! value = 869.0
  GENERATION 8: new record! value = 889.0
Value: 889.0


In [20]:
#PROBLEM2
GENERATIONS = 50
POPULATION_SIZE = 20  # mu (dim of population)
OFFSPRING_SIZE = 200  # lambda (num of children per generation)

best_solution, best_score = my_algorithm(
    generations=GENERATIONS,
    mu=POPULATION_SIZE,
    lambda_=OFFSPRING_SIZE, 
    num_knapsacks=NUM_KNAPSACKS2,
    num_items=NUM_ITEMS2, 
    weights=WEIGHTS2, 
    constraints=CONSTRAINTS2, 
    values=VALUES2
)

if best_score <= 0:
    print("no solution found")
else:
    print("Value:", best_score)

  GENERATION 1: new record! value = 19827.0
  GENERATION 2: new record! value = 20251.0
  GENERATION 5: new record! value = 20375.0
  GENERATION 8: new record! value = 20685.0
  GENERATION 9: new record! value = 21082.0
  GENERATION 11: new record! value = 21348.0
  GENERATION 15: new record! value = 21735.0
  GENERATION 17: new record! value = 21799.0
  GENERATION 18: new record! value = 22237.0
  GENERATION 20: new record! value = 22401.0
  GENERATION 23: new record! value = 22797.0
  GENERATION 26: new record! value = 23169.0
  GENERATION 30: new record! value = 23253.0
  GENERATION 31: new record! value = 23630.0
  GENERATION 32: new record! value = 23792.0
  GENERATION 34: new record! value = 24002.0
  GENERATION 35: new record! value = 24015.0
  GENERATION 37: new record! value = 24259.0
  GENERATION 40: new record! value = 24734.0
  GENERATION 43: new record! value = 25596.0
  GENERATION 45: new record! value = 25724.0
Value: 25724.0


In [21]:
#PROBLEM3
initial_solution = create_random_valid_solution(NUM_KNAPSACKS3, NUM_ITEMS3, WEIGHTS3, CONSTRAINTS3)
best_solution, best_score = simulated_annealing_fast(
    initial_solution=initial_solution,
    max_steps=1000,
    initial_temp=100.0,
    cooling_rate=0.995,
    values=VALUES3,
    num_knapsacks=NUM_KNAPSACKS3,
    weights=WEIGHTS3,
    constraints=CONSTRAINTS3,
    num_items=NUM_ITEMS3
)
if best_score <= 0:
    print("no solution found")
else:
    print("Value:", best_score)

Value: 807085.0


In [22]:
"""
OBSERVATIONS: 
Regarding problems 1 and 2, the developed solution is definitely much better than a standard simulated annealing, and in terms of performance/cost, it is worth using it.
However, for the third problem, generating a valid solution is computationally expensive (about 20 seconds per solution), so the algorithm—which needs to generate many of them—becomes unusable in terms of time.
Therefore, we tried two approaches: starting from an invalid solution with our algorithm, and using a standard simulated annealing while trying to optimize it starting from a random but valid initial solution.
When evaluating in terms of cost and performance, the second approach makes more sense.
"""


'\nOBSERVATIONS: \nRegarding problems 1 and 2, the developed solution is definitely much better than a standard simulated annealing, and in terms of performance/cost, it is worth using it.\nHowever, for the third problem, generating a valid solution is computationally expensive (about 20 seconds per solution), so the algorithm—which needs to generate many of them—becomes unusable in terms of time.\nTherefore, we tried two approaches: starting from an invalid solution with our algorithm, and using a standard simulated annealing while trying to optimize it starting from a random but valid initial solution.\nWhen evaluating in terms of cost and performance, the second approach makes more sense.\n'